In [1]:
cd ..

/workspace/llm-graph-construction


/opt/conda/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
!pip install -q seqeval

In [3]:
!pip install wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 83.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 14.3 MB/s eta 0:00:00


In [4]:
from training.train_event_extraction import *

/workspace/llm-graph-construction


In [5]:
test_name = "thyme"

dataset_loader_balanced = lambda: load_stored_dataset_combination_graph(balanced=True, dataset=test_name)
dataset_loader_unbalanced = lambda: load_stored_dataset_combination_graph(balanced=False, dataset=test_name)

if test_name == "thyme":
    number_of_relations = 11 # imamo relacije 0, 2, 3, 5, 6, 7, 8, 9, 10
else:
    number_of_relations = 3

In [6]:
def load_stored_dataset_combination_graph(balanced=True, dataset="i2b2"):
    dataset_train = DFDataset()
    dataset_train.load("pregenerated/"+dataset+"_dataset_train_rawkg.pt")
    dataset_val = DFDataset()
    dataset_val.load("pregenerated/"+dataset+"_dataset_val_rawkg.pt")
    dataset_test = DFDataset()
    dataset_test.load("pregenerated/"+dataset+"_dataset_test_rawkg.pt")

    if balanced:
        dataset_train.oversample_pregenerated()
        dataset_val.oversample_pregenerated()

    return dataset_train, dataset_val, dataset_test

In [7]:
dataset = load_stored_dataset_combination_graph(balanced=False, dataset=test_name)

/opt/conda/lib/python3.10/site-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /opt/conda/lib/python3.10/site-packages/libpyg.so: undefined symbol: _ZN2at4_ops10zeros_like4callERKNS_6TensorEN3c108optionalINS5_10ScalarTypeEEENS6_INS5_6LayoutEEENS6_INS5_6DeviceEEENS6_IbEENS6_INS5_12MemoryFormatEEE
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/opt/conda/lib/python3.10/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /opt/conda/lib/python3.10/site-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/opt/conda/lib/python3.10/site-packages/torch_geometric/typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: /opt/conda/lib/python3.10/site

In [8]:
dataset_train, dataset_val, dataset_test = dataset

In [9]:
df = dataset_train.df

In [35]:
def train_event_extraction(df_train, df_val, df_test):
    seqeval = evaluate.load("seqeval")

    # df = combining_data.read_i2b2(full_text=True, use_test_files=False, include_rows_without_absolute=True)
    df_train = convert_relation_extraction_df_to_event_extraction(df_train, tokenizer)
    df_val = convert_relation_extraction_df_to_event_extraction(df_val, tokenizer)
    df_test = convert_relation_extraction_df_to_event_extraction(df_test, tokenizer)
    # border1 = int(0.7 * len(df_events))
    # border2 = int((0.7 + 0.2) * len(df_events))
    # df_train, df_val, df_test = np.split(df_events, [border1, border2])

    dataset_train = DFDataset(df_train, lambda row, args: {"besedilo":row["text"], "labels":row["labels"], "tokens": row["tokens"]}, {})
    dataset_val = DFDataset(df_val, lambda row, args: {"besedilo":row["text"], "labels":row["labels"], "tokens": row["tokens"]}, {})

    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    print(device)
    model = EventExtraction(tokenizer)
    model.to(device)
    # model = MultiModalPrediction(number_of_relations=3, combine_embeddings=True)

    training_args = TrainingArguments(
        output_dir="./results",
        learning_rate=0.1,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        auto_find_batch_size=True,
        num_train_epochs=50,
        weight_decay=0.01,
        gradient_accumulation_steps=1,
        evaluation_strategy="epoch",
        logging_strategy="epoch",
        push_to_hub=False,
    )
    # "adamw_hf", "adamw_torch", "adamw_torch_fused", "adamw_apex_fused", "adamw_anyprecision" or "adafactor"
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset_train,
        eval_dataset=dataset_val,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    torch.save(model, "event-model2.pt")
    pass

In [36]:
train_event_extraction(dataset_train.df, dataset_val.df, dataset_test.df)

loading configuration file ./pretrained models/PubmedBERTbase-MimicBig-EntityBERT/config.json
Model config BertConfig {
  "_name_or_path": "./pretrained models/PubmedBERTbase-MimicBig-EntityBERT",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.20.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30539
}

loading weights file ./pretrained models/PubmedBERTbase-MimicBig-EntityBERT/pytorch_model.bin


cuda:0


Some weights of the model checkpoint at ./pretrained models/PubmedBERTbase-MimicBig-EntityBERT were not used when initializing BertModel: ['cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of BertModel were initialized from the model checkpoint at ./pretrained models/PubmedBERTbase-MimicBig-EntityBERT.
If your task is

Epoch,Training Loss,Validation Loss,Precision
1,1.167500,1.217474,0.100998
2,1.185200,1.217474,0.100998
3,1.186100,1.217474,0.100998
4,1.185200,1.217474,0.100998
5,1.184100,1.217474,0.100998
6,1.185300,1.217474,0.100998
7,1.186100,1.217474,0.100998
8,1.186000,1.217474,0.100998
9,1.185900,1.217474,0.100998
10,1.186600,1.217474,0.100998


***** Running Evaluation *****
  Num examples = 48
  Batch size = 4
The following columns in the evaluation set don't have a corresponding argument in `EventExtraction.forward` and have been ignored: besedilo. If besedilo are not expected by `EventExtraction.forward`,  you can safely ignore this message.


...

***** Running Evaluation *****
  Num examples = 48
  Batch size = 4
The following columns in the evaluation set don't have a corresponding argument in `EventExtraction.forward` and have been ignored: besedilo. If besedilo are not expected by `EventExtraction.forward`,  you can safely ignore this message.


{'precision': 0.10099783080260304}


***** Running Evaluation *****
  Num examples = 48
  Batch size = 4
The following columns in the evaluation set don't have a corresponding argument in `EventExtraction.forward` and have been ignored: besedilo. If besedilo are not expected by `EventExtraction.forward`,  you can safely ignore this message.


{'precision': 0.10099783080260304}




Training completed. Do not forget to share your model on huggingface.co/models =)




In [21]:
a = convert_relation_extraction_df_to_event_extraction(dataset_train.df, tokenizer)

In [34]:
all_sum = {False: 0, True: 0}
for l in a["labels"]:
    count = Counter(l)
    all_sum[False] += count[False]
    all_sum[True] += count[True]
print(all_sum)
print(all_sum[False] / (all_sum[False] + all_sum[True]))

{False: 56948, True: 8502}
0.8700993124522536


In [19]:
from collections import Counter
print(Counter(dataset_val.df["class"]))

Counter({'CONTAINS': 2375, 'OVERLAP': 490, 'BEFORE': 457, 'BEGINS-ON': 95, 'ENDS-ON': 58, 'TERMINATES': 40, 'INITIATES': 39, 'CONTINUES': 26, 'REINITIATES': 5})


In [15]:
# code injection

import torch
from torch import nn
from transformers import AutoModel, AutoTokenizer

class EventExtraction(nn.Module):
    def __init__(self, tokenizer):
        super(EventExtraction, self).__init__()
        self.bert = AutoModel.from_pretrained("./pretrained models/PubmedBERTbase-MimicBig-EntityBERT")
        self.classification = nn.Linear(768, 2)
        self.softmax = nn.Softmax(dim=1)
        self.tokenizer = tokenizer
        self.loss = nn.CrossEntropyLoss()

    def forward(self, tokens, labels=None):
        embeddings = self.bert(**tokens)
        embeddings = embeddings.last_hidden_state
        result = []
        batched_result = []
        truth = []
        for sample in range(len(embeddings)):
            # result.append([])
            batched_result.append([])
            # truth.append([])
            for token in range(len(embeddings[sample])):
                token_embedding = embeddings[sample][token]
                logits = self.classification(token_embedding)
                # result[sample].append(logits)
                batched_result[sample].append(logits)
                result.append(logits)
                # truth[sample].append(1 if labels[sample][token] else 0)
                if labels is not None:
                    truth.append(1 if labels[sample][token] else 0)
            # result[sample] = torch.stack(result[sample], dim=0)
            batched_result[sample] = self.softmax(torch.stack(batched_result[sample], dim=0))
        batched_result = torch.stack(batched_result, dim=0)
        result = torch.stack(result, dim=0)
        result = self.softmax(result)
        if labels is not None:
            truth = torch.tensor(truth, device=result.device)
            loss = self.loss(result, truth)
            return {"loss": loss, "results": result, "batched_result": batched_result, "truth": truth, "mask": tokens["attention_mask"]}
        else:
            return {"results": result, "batched_result": batched_result, "truth": truth, "mask": tokens["attention_mask"]}
